# Unsupervised Clustering and Statistical Framework for TME Characterization## Application to Early-Stage NSCLC from the ALCHEMIST Trial**Authors:** Elizabeth Baker¹, Nathan Mehaffy²  ¹ Department of Bioengineering, University of Louisville  ² Department of Electrical and Computer Engineering, Carnegie Mellon University**Paper:** *Computers in Biology and Medicine* (2026)  **Data:** NCI Genomic Data Commons, ALCHEMIST-ALCH, Data Release 45 (December 4, 2025)---### Pipeline Overview| Module | Method | Purpose ||--------|--------|---------|| 1 | Mann-Whitney U / Kruskal-Wallis | Nonparametric association testing || 2 | Benjamini-Hochberg FDR | Multiple testing correction (59 tests) || 3 | Spearman rank correlation | Coordinated infiltration patterns || 4 | K-means (silhouette-optimized) | Unsupervised TME subtype discovery |

## 1. Setup and Data Loading

In [ ]:
import pandas as pdimport numpy as npfrom scipy.stats import mannwhitneyu, kruskal, spearmanr, chi2_contingencyfrom statsmodels.stats.multitest import multipletestsfrom sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeansfrom sklearn.metrics import silhouette_scoreimport matplotlib.pyplot as pltimport matplotlibimport seaborn as snsimport warningswarnings.filterwarnings('ignore')matplotlib.rcParams.update({    'font.size': 11, 'figure.dpi': 150,    'font.family': 'sans-serif', 'axes.spines.top': False, 'axes.spines.right': False})print("Dependencies loaded successfully.")

## 2. Data PreprocessingLoad clinical, slide, and sample data from GDC ALCHEMIST-ALCH.  Convert placeholder values (`'--`) to NaN, merge metadata, derive age and histology groups.

In [ ]:
# --- Load raw data ---DATA_DIR = "/kaggle/input/datasets/elizabethbaker28"clin = pd.read_csv(f"{DATA_DIR}/alchemistclinical/clinical.tsv", sep='\t')slide = pd.read_csv(f"{DATA_DIR}/alchemistbiospecimen/slide.tsv", sep='\t')sample = pd.read_csv(f"{DATA_DIR}/alchemistbiospecimen/sample.tsv", sep='\t')# --- Clean clinical data ---clin_clean = clin[['cases.case_id', 'cases.submitter_id', 'demographic.gender',                    'diagnoses.age_at_diagnosis', 'diagnoses.primary_diagnosis']].copy()clin_clean.columns = ['case_id', 'submitter_id', 'gender', 'age_at_diagnosis_days', 'histology']clin_clean['age_at_diagnosis_days'] = pd.to_numeric(    clin_clean['age_at_diagnosis_days'].replace("'--", np.nan), errors='coerce')clin_clean['age_years'] = clin_clean['age_at_diagnosis_days'] / 365.25clin_clean['age_group'] = pd.cut(    clin_clean['age_years'], bins=[0, 55, 65, 75, 100], labels=['≤55', '56-65', '66-75', '>75'])# --- Simplify histology ---def classify_histology(h):    h = str(h).lower()    if 'adenosquamous' in h: return 'Adenosquamous'    if 'squamous' in h: return 'Squamous'    if any(x in h for x in ['adenocarcinoma', 'mucinous', 'lepidic', 'acinar']): return 'Adenocarcinoma'    if 'large cell' in h: return 'Large Cell'    return 'Other NSCLC'clin_clean['histology_group'] = clin_clean['histology'].apply(classify_histology)# --- Clean slide TME data ---TME_COLS_RAW = [    'slides.percent_lymphocyte_infiltration', 'slides.percent_monocyte_infiltration',    'slides.percent_necrosis', 'slides.percent_neutrophil_infiltration',    'slides.percent_normal_cells', 'slides.percent_stromal_cells',    'slides.percent_tumor_cells', 'slides.percent_tumor_nuclei']TME_COLS = ['lymphocyte_pct', 'monocyte_pct', 'necrosis_pct', 'neutrophil_pct',            'normal_cells_pct', 'stromal_pct', 'tumor_cells_pct', 'tumor_nuclei_pct']slide_clean = slide[['cases.case_id', 'samples.sample_id'] + TME_COLS_RAW].copy()slide_clean.columns = ['case_id', 'sample_id'] + TME_COLSfor col in TME_COLS:    slide_clean[col] = pd.to_numeric(slide_clean[col].replace("'--", np.nan), errors='coerce')# Filter slides with actual TME dataslide_valid = slide_clean.dropna(subset=TME_COLS, how='all')slide_valid = slide_valid[slide_valid[TME_COLS].sum(axis=1) > 0]print(f"Slides with TME data: {len(slide_valid)}")# Average per patientpatient_tme = slide_valid.groupby('case_id')[TME_COLS].mean().reset_index()# --- Merge clinical + TME ---df = clin_clean.merge(patient_tme, on='case_id', how='inner')print(f"Patients with clinical + TME data: {len(df)}")print(f"Histology: {dict(df['histology_group'].value_counts())}")print(f"Gender: {dict(df['gender'].value_counts())}")print(f"Age: median={df['age_years'].median():.1f} (IQR: {df['age_years'].quantile(0.25):.1f}-{df['age_years'].quantile(0.75):.1f})")# --- Prepare slide-level data for primary vs recurrence ---sample_type = sample[['samples.sample_id', 'samples.tumor_descriptor']].copy()sample_type.columns = ['sample_id', 'tumor_descriptor']sample_type['tumor_descriptor'] = sample_type['tumor_descriptor'].replace("'--", np.nan)slide_with_descriptor = slide_clean.merge(sample_type, on='sample_id', how='left')

## 3. Module 1 — Association Testing + Module 2 — FDR CorrectionExecute all 59 hypothesis tests, then apply Benjamini-Hochberg FDR correction across the full family.

In [ ]:
all_tests = []# --- Adenocarcinoma vs Squamous (8 Mann-Whitney) ---adeno = df[df['histology_group'] == 'Adenocarcinoma']squam = df[df['histology_group'] == 'Squamous']for col in TME_COLS:    stat, p = mannwhitneyu(adeno[col].dropna(), squam[col].dropna(), alternative='two-sided')    all_tests.append(('Histology_' + col, p))# --- Kruskal-Wallis across all histology groups (8) ---for col in TME_COLS:    groups = [g[col].dropna().values for _, g in df.groupby('histology_group') if len(g[col].dropna()) >= 5]    if len(groups) >= 2:        stat, p = kruskal(*groups)        all_tests.append(('KW_Histology_' + col, p))# --- Gender (8 Mann-Whitney) ---female = df[df['gender'] == 'female']male = df[df['gender'] == 'male']for col in TME_COLS:    stat, p = mannwhitneyu(female[col].dropna(), male[col].dropna(), alternative='two-sided')    all_tests.append(('Gender_' + col, p))# --- Age groups Kruskal-Wallis (8) ---for col in TME_COLS:    groups = [df[df['age_group'] == ag][col].dropna().values              for ag in ['≤55', '56-65', '66-75', '>75']              if len(df[df['age_group'] == ag][col].dropna()) >= 5]    stat, p = kruskal(*groups)    all_tests.append(('KW_Age_' + col, p))# --- Age continuous Spearman (8) ---for col in TME_COLS:    valid = df[['age_years', col]].dropna()    rho, p = spearmanr(valid['age_years'], valid[col])    if hasattr(p, '__len__'): p = float(p.flat[0])    all_tests.append(('Spearman_Age_' + col, p))# --- Immune status associations ---t1 = df['lymphocyte_pct'].dropna().quantile(0.33)t2 = df['lymphocyte_pct'].dropna().quantile(0.67)df['immune_status'] = pd.cut(df['lymphocyte_pct'], bins=[-1, t1, t2, 101],                              labels=['Cold', 'Intermediate', 'Hot'])df_imm = df.dropna(subset=['immune_status'])ct = pd.crosstab(df_imm['immune_status'], df_imm['histology_group'])_, p, _, _ = chi2_contingency(ct)all_tests.append(('ImmuneStatus_vs_Histology', p))ct_g = pd.crosstab(df_imm['immune_status'], df_imm['gender'])_, p, _, _ = chi2_contingency(ct_g)all_tests.append(('ImmuneStatus_vs_Gender', p))for col in [c for c in TME_COLS if c != 'lymphocyte_pct']:    groups = [df_imm[df_imm['immune_status'] == s][col].dropna().values for s in ['Cold', 'Intermediate', 'Hot']]    stat, p = kruskal(*groups)    all_tests.append(('ImmuneStatus_' + col, p))# --- Primary vs Recurrent (8) ---primary = slide_with_descriptor[slide_with_descriptor['tumor_descriptor'] == 'Primary']recurrent = slide_with_descriptor[slide_with_descriptor['tumor_descriptor'] == 'Recurrence']for col in TME_COLS:    p_vals = primary[col].dropna()    r_vals = recurrent[col].dropna()    if len(r_vals) >= 5:        stat, p = mannwhitneyu(p_vals, r_vals, alternative='two-sided')        all_tests.append(('Recurrence_' + col, p))# --- Cluster associations (computed after clustering, placeholder) ---# Will be added after Module 4print(f"Tests collected (pre-clustering): {len(all_tests)}")

## 4. Module 3 — Spearman Correlation Analysis

In [ ]:
print("Spearman Correlation Matrix (TME components):\n")corr_matrix = pd.DataFrame(index=TME_COLS, columns=TME_COLS, dtype=float)for c1 in TME_COLS:    for c2 in TME_COLS:        valid = df[[c1, c2]].dropna()        rho, _ = spearmanr(valid[c1], valid[c2])        if hasattr(rho, '__len__'): rho = float(rho.flat[0])        corr_matrix.loc[c1, c2] = rhoshort_names = [c.replace('_pct', '') for c in TME_COLS]corr_display = corr_matrix.copy()corr_display.index = short_namescorr_display.columns = short_namesprint(corr_display.round(2).to_string())print("\n--- Immune co-infiltration module ---")pairs = [('monocyte_pct', 'neutrophil_pct'), ('lymphocyte_pct', 'monocyte_pct'),         ('lymphocyte_pct', 'neutrophil_pct')]for c1, c2 in pairs:    valid = df[[c1, c2]].dropna()    rho, p = spearmanr(valid[c1], valid[c2])    print(f"  {c1} vs {c2}: rho={rho:.2f}, p={p:.2e}")

## 5. Module 4 — Silhouette-Optimized K-Means Clustering

In [ ]:
df_complete = df.dropna(subset=TME_COLS).copy()X = df_complete[TME_COLS].valuesscaler = StandardScaler()X_scaled = scaler.fit_transform(X)# Evaluate k=2 through k=6print("Silhouette scores:")sil_scores = {}for k in range(2, 7):    km = KMeans(n_clusters=k, random_state=42, n_init=10)    labels = km.fit_predict(X_scaled)    sil = silhouette_score(X_scaled, labels)    sil_scores[k] = sil    print(f"  k={k}: silhouette={sil:.3f}")# Select k=6 for biological interpretability (see Methods)K_OPT = 6km_final = KMeans(n_clusters=K_OPT, random_state=42, n_init=10)df_complete['tme_cluster'] = km_final.fit_predict(X_scaled)print(f"\nSelected k={K_OPT} (silhouette={sil_scores[K_OPT]:.3f})")print(f"\n--- Cluster profiles ---")for c in sorted(df_complete['tme_cluster'].unique()):    cl = df_complete[df_complete['tme_cluster'] == c]    n = len(cl)    pct = 100 * n / len(df_complete)    print(f"\nCluster {c}: n={n} ({pct:.1f}%)")    for col in TME_COLS:        print(f"  {col}: median={cl[col].median():.1f}")    hist = dict(cl['histology_group'].value_counts())    print(f"  Histology: {hist}")# Add cluster chi-square tests to the test familyct_cl = pd.crosstab(df_complete['tme_cluster'], df_complete['histology_group'])_, p, _, _ = chi2_contingency(ct_cl)all_tests.append(('Cluster_vs_Histology', p))ct_clg = pd.crosstab(df_complete['tme_cluster'], df_complete['gender'])_, p, _, _ = chi2_contingency(ct_clg)all_tests.append(('Cluster_vs_Gender', p))print(f"\nTotal tests for FDR correction: {len(all_tests)}")

## 6. Apply Benjamini-Hochberg FDR Correction (All 59 Tests)

In [ ]:
test_names = [t[0] for t in all_tests]raw_pvals = np.array([t[1] for t in all_tests])reject, q_values, _, _ = multipletests(raw_pvals, alpha=0.05, method='fdr_bh')print(f"{'Test':<45} {'Raw p':>12} {'FDR q':>12} {'Sig':>6}")print("-" * 80)for name, raw_p, q, sig in sorted(zip(test_names, raw_pvals, q_values, reject), key=lambda x: x[1]):    sig_str = "***" if q < 0.001 else "**" if q < 0.01 else "*" if q < 0.05 else "ns"    print(f"{name:<45} {raw_p:>12.2e} {q:>12.2e} {sig_str:>6}")print(f"\nTotal tests: {len(all_tests)}")print(f"Significant (q<0.05): {sum(reject)}")print(f"Not significant: {sum(~reject)}")print(f"Discovery rate: {100*sum(reject)/len(reject):.1f}%")

## 7. Figure Generation

In [ ]:
TME_LABELS = {    'lymphocyte_pct': 'Lymphocytes (%)', 'monocyte_pct': 'Monocytes (%)',    'necrosis_pct': 'Necrosis (%)', 'neutrophil_pct': 'Neutrophils (%)',    'normal_cells_pct': 'Normal Cells (%)', 'stromal_pct': 'Stromal Cells (%)',    'tumor_cells_pct': 'Tumor Cells (%)', 'tumor_nuclei_pct': 'Tumor Nuclei (%)'}# ---- FIGURE 1: TME by histology ----fig, axes = plt.subplots(2, 4, figsize=(18, 9))for i, col in enumerate(TME_COLS):    ax = axes.flatten()[i]    data_plot = df[df['histology_group'].isin(['Adenocarcinoma', 'Squamous'])]    sns.boxplot(data=data_plot, x='histology_group', y=col, ax=ax,                palette={'Adenocarcinoma': '#4C72B0', 'Squamous': '#DD8452'}, showfliers=False)    a, s = adeno[col].dropna(), squam[col].dropna()    _, p = mannwhitneyu(a, s, alternative='two-sided')    ax.set_title(f"{TME_LABELS[col]}\np={'%.1e'%p}", fontsize=10)    ax.set_xlabel(''); ax.set_ylabel('')plt.suptitle('Fig. 1: TME Composition by Histological Subtype', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('fig1_tme_by_histology.png', dpi=300, bbox_inches='tight')plt.show()# ---- FIGURE 2: Correlation heatmap ----fig, ax = plt.subplots(figsize=(10, 8))mask = np.triu(np.ones_like(corr_matrix, dtype=bool))sns.heatmap(corr_matrix.astype(float), mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',            center=0, vmin=-1, vmax=1, xticklabels=short_names, yticklabels=short_names, ax=ax)ax.set_title('Fig. 2: Spearman Correlation Matrix of TME Components', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('fig2_tme_correlations.png', dpi=300, bbox_inches='tight')plt.show()# ---- FIGURE 3: Primary vs Recurrent ----fig, axes = plt.subplots(1, 4, figsize=(16, 4))highlight = ['normal_cells_pct', 'lymphocyte_pct', 'tumor_cells_pct', 'tumor_nuclei_pct']for i, col in enumerate(highlight):    ax = axes[i]    p_v = primary[col].dropna(); r_v = recurrent[col].dropna()    ax.boxplot([p_v, r_v], labels=['Primary', 'Recurrent'], showfliers=False)    _, p = mannwhitneyu(p_v, r_v, alternative='two-sided')    ax.set_title(f"{TME_LABELS[col]}\np={'%.1e'%p}", fontsize=10)plt.suptitle('Fig. 3: Primary vs Recurrent TME', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('fig3_primary_vs_recurrent.png', dpi=300, bbox_inches='tight')plt.show()# ---- FIGURE 4: TME clusters ----fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [2, 1]})cluster_medians = df_complete.groupby('tme_cluster')[TME_COLS].median()cluster_medians.columns = short_namessns.heatmap(cluster_medians, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax1)ax1.set_title('Median TME Values per Cluster')ax1.set_ylabel('Cluster')hist_by_cluster = pd.crosstab(df_complete['tme_cluster'], df_complete['histology_group'], normalize='index')hist_by_cluster.plot(kind='barh', stacked=True, ax=ax2,                      color=['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3'])ax2.set_title('Histological Composition')ax2.set_xlabel('Proportion')ax2.legend(loc='lower right', fontsize=8)plt.suptitle('Fig. 4: Six TME Subtypes (K-means, k=6)', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('fig4_tme_clusters.png', dpi=300, bbox_inches='tight')plt.show()# ---- FIGURE 5: Immune status ----fig, axes = plt.subplots(2, 4, figsize=(18, 9))for i, col in enumerate(TME_COLS):    ax = axes.flatten()[i]    data_plot = df.dropna(subset=['immune_status'])    sns.boxplot(data=data_plot, x='immune_status', y=col, ax=ax,                order=['Cold', 'Intermediate', 'Hot'],                palette={'Cold': '#6495ED', 'Intermediate': '#FFD700', 'Hot': '#FF4500'}, showfliers=False)    ax.set_title(TME_LABELS[col], fontsize=10)    ax.set_xlabel(''); ax.set_ylabel('')plt.suptitle('Fig. 5: TME Composition by Immune Status', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('fig5_immune_status.png', dpi=300, bbox_inches='tight')plt.show()print("All figures saved.")

## 8. Summary| Finding | FDR q-value | Status ||---------|-------------|--------|| Necrosis: Squamous > Adeno | 1.5×10⁻²⁸ | ✅ Significant || Neutrophils: Squamous > Adeno | 1.3×10⁻³ | ✅ Significant || Lymphocytes: Squamous vs Adeno | 0.69 | ❌ Not significant || Any immune cell vs gender | >0.22 | ❌ Not significant || Any TME vs age | >0.15 | ❌ Not significant || Normal cells: Recurrent < Primary | 4.0×10⁻⁷ | ✅ Significant || Lymphocytes: Recurrent vs Primary | 0.71 | ❌ Not significant || Cluster vs Histology | 4.0×10⁻⁷ | ✅ Significant |**17/59 tests significant after FDR correction (28.8% discovery rate)**---*Code: [GitHub Repository](https://github.com/elizabethbaker28/alchemist-tme-analysis)*  *Data: [NCI GDC ALCHEMIST-ALCH](https://portal.gdc.cancer.gov)*  *Paper: Baker E, Mehaffy N. Computers in Biology and Medicine (2026)*